# Feature Engineering

## Objetivo

Construir las estructuras necesarias para entrenar sistemas de recomendación basados en interacciones entre usuarios y productos.

En esta etapa se generarán las variables y matrices que permitirán modelar las preferencias de los usuarios.

In [1]:
# Importar librerías necesarias

import pandas as pd
import numpy as np

In [2]:
# Cargar dataset procesado

interactions = pd.read_parquet(
    "../data/processed/interactions_filtered.parquet"
)

interactions.head()

,user_id,product_id,reordered,product_name,department,aisle
0,202279,33120,1,Organic Egg Whites,dairy eggs,eggs
1,202279,28985,1,Michigan Organic Kale,produce,fresh vegetables
2,202279,9327,0,Garlic Powder,pantry,spices seasonings
3,202279,45918,1,Coconut Butter,pantry,oils vinegars
4,202279,30035,0,Natural Sweetener,pantry,baking ingredients


In [3]:
# Revisar dimensiones

print("Interacciones:", len(interactions))
print("Usuarios:", interactions["user_id"].nunique())
print("Productos:", interactions["product_id"].nunique())

Interacciones: 32300077
Usuarios: 206208
Productos: 35922


In [4]:
# Calcular cantidad de compras por usuario

user_features = (
    interactions
    .groupby("user_id")
    .agg(
        total_interactions=("product_id", "count"),
        unique_products=("product_id", "nunique"),
        reorder_rate=("reordered", "mean")
    )
    .reset_index()
)

user_features.head()

,user_id,total_interactions,unique_products,reorder_rate
0,1,59,18,0.694915
1,2,195,102,0.476923
2,3,88,33,0.625000
3,4,17,16,0.058824
4,5,37,23,0.378378


In [5]:
# Estadísticas descriptivas de usuarios

user_features.describe()

,user_id,total_interactions,unique_products,reorder_rate
count,206208.000000,206208.000000,206208.000000,206208.000000
mean,103105.178276,156.638331,64.041332,0.433091
std,59527.644457,203.739244,56.132415,0.212253
min,1.000000,2.000000,1.000000,0.000000
25%,51552.750000,39.000000,25.000000,0.269231
50%,103105.500000,83.000000,47.000000,0.430303
75%,154657.250000,187.000000,85.000000,0.596964
max,206209.000000,3725.000000,724.000000,0.989529


In [6]:
# Construir métricas de producto

product_features = (
    interactions
    .groupby("product_id")
    .agg(
        total_purchases=("user_id", "count"),
        unique_users=("user_id", "nunique"),
        reorder_rate=("reordered", "mean")
    )
    .reset_index()
)

product_features.head()

,product_id,total_purchases,unique_users,reorder_rate
0,1,1852,716,0.613391
1,2,90,78,0.133333
2,3,277,74,0.732852
3,4,329,182,0.446809
4,7,30,18,0.400000


In [7]:
# Estadísticas descriptivas de productos

product_features.describe()

,product_id,total_purchases,unique_users,reorder_rate
count,35922.000000,35922.000000,35922.000000,35922.000000
mean,24848.086075,899.172568,367.625271,0.429594
std,14337.507434,5615.951732,1527.388972,0.178498
min,1.000000,20.000000,2.000000,0.000000
25%,12519.500000,48.000000,29.000000,0.300000
50%,24807.500000,126.000000,71.000000,0.441757
75%,37248.750000,443.000000,226.000000,0.564588
max,49688.000000,472565.000000,73956.000000,0.941176


In [9]:
# Usuarios con al menos 50 interacciones

active_users = user_features[
    user_features["total_interactions"] >= 50
]["user_id"]

len(active_users)

138882

In [10]:
# Dataset reducido para modelado inicial

model_data = interactions[
    interactions["user_id"].isin(active_users)
].copy()

model_data.shape

(30431675, 6)

## Selección de usuarios activos

Para la construcción de los modelos se seleccionaron usuarios con al menos 50 interacciones históricas.

Esta decisión permite reducir ruido, mantener usuarios con suficiente información para generar recomendaciones y disminuir la sparsity del problema.

Luego se evaluarán diferentes subconjuntos de entrenamiento según los requerimientos computacionales de cada modelo.

In [13]:
# Guardar dataset para modelado

model_data.to_parquet(
    "../data/processed/model_data.parquet",
    index=False
)

In [14]:
# Verificar que el archivo fue creado correctamente

import os

os.path.exists(
    "../data/processed/model_data.parquet"
)

True

# Conclusiones de Feature Engineering

## Features generadas

Se construyeron métricas descriptivas para usuarios y productos:

### Usuarios
- Total de interacciones.
- Cantidad de productos únicos consumidos.
- Tasa de recompra.

### Productos
- Cantidad total de compras.
- Cantidad de usuarios únicos.
- Tasa de recompra.

## Selección de usuarios activos

Para la construcción de los modelos se seleccionaron usuarios con al menos 50 interacciones históricas.

Esta decisión permite reducir ruido, mantener usuarios con suficiente información para generar recomendaciones y disminuir la sparsity del problema.

## Dataset para modelado

Se generó un dataset consolidado de interacciones usuario-producto que servirá como base para los modelos de recomendación.